# 03: Pretraining and optimization

[Course index](../README_en.md) | [中文版本](../notebooks/03_pretraining.ipynb)

**Goal:** understand the data window, AdamW, learning-rate schedule, mixed precision, accumulation, and validation.


## 1. Data pipeline

The pipeline is `clean text → tokenizer → token stream → fixed windows`. A window of `block_size + 1` tokens becomes shifted input/target tensors.

```bash
python scripts/prepare_pretrain_data.py
python train/pretrain.py --config configs/train_pretrain.yaml
```

With micro-batch 16, accumulation 8, and length 256, one optimizer update processes $16	imes8	imes256=32768$ token positions.


## 2. AdamW

$$m_t=eta_1m_{t-1}+(1-eta_1)g_t,\qquad v_t=eta_2v_{t-1}+(1-eta_2)g_t^2,$$

$$	heta_{t+1}=(1-\eta_t\lambda)	heta_t-\eta_trac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$$

AdamW decouples weight decay from the adaptive gradient update.


### Hands-on check

Perform one AdamW update on a scalar parameter.


In [ ]:
import torch

parameter = torch.nn.Parameter(torch.tensor([1.0]))
optimizer_demo = torch.optim.AdamW([parameter], lr=0.1, weight_decay=0.01)
loss_demo = (parameter - 0.0).pow(2).sum()
loss_demo.backward()
before = parameter.item()
optimizer_demo.step()
print(f"parameter: {before:.4f} -> {parameter.item():.4f}")
assert parameter.item() < before


## 3. Warmup and cosine decay

Warmup increases the learning rate linearly; cosine decay then approaches a non-zero minimum:

$$\eta_t=\eta_{min}+	frac12(\eta_{max}-\eta_{min})[1+\cos(\pi r_t)].$$


### Hands-on check

Compute a small warmup-plus-cosine schedule and inspect its endpoints.


In [ ]:
import math

def learning_rate(step, warmup=4, total=20, high=3e-4, low=3e-5):
    if step < warmup:
        return high * (step + 1) / warmup
    ratio = (step - warmup) / max(1, total - warmup - 1)
    return low + 0.5 * (high - low) * (1 + math.cos(math.pi * ratio))

lrs = [learning_rate(step) for step in range(20)]
print("warmup:", [f"{value:.2e}" for value in lrs[:4]])
print("last:", f"{lrs[-1]:.2e}")
assert lrs[3] == max(lrs) and abs(lrs[-1] - 3e-5) < 1e-10


## 4. Mixed precision, accumulation, and clipping

FP16 lowers memory use; `GradScaler` protects small gradients. For $K$ micro-batches, divide every loss by $K$ and call `optimizer.step()` only after accumulation. Global clipping applies

$$g\leftarrow g\min\left(1,rac{c}{\lVert gVert_2}ight).$$


### Hands-on check

Run four micro-steps with an optimizer update every two steps.


In [ ]:
import torch

# CPU-sized gradient-accumulation check; formal training uses train/pretrain.py.
torch.manual_seed(0)
layer = torch.nn.Linear(4, 2)
optimizer = torch.optim.AdamW(layer.parameters(), lr=1e-3)
accum_steps = 2
optimizer.zero_grad(set_to_none=True)

losses = []
for micro_step in range(4):
    x = torch.randn(3, 4)
    target = torch.randn(3, 2)
    loss = torch.nn.functional.mse_loss(layer(x), target)
    (loss / accum_steps).backward()
    losses.append(loss.item())
    if (micro_step + 1) % accum_steps == 0:
        grad_norm = torch.nn.utils.clip_grad_norm_(layer.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        print(f"update={(micro_step + 1) // accum_steps}, grad_norm={grad_norm:.4f}")

print("micro losses:", [round(value, 4) for value in losses])


## 5. How to tell whether pretraining works

Track train/validation NLL, PPL, fixed-prompt samples, repetition, sentence endings, checkpoint recovery, and the train–validation gap. A low loss can still coexist with repetition; one fluent sample can be luck.

- [Previous](02_transformer.ipynb) · [Index](../README_en.md) · [Next](04_sft.ipynb)
- [`train/pretrain.py`](../../../train/pretrain.py) · [`train/common.py`](../../../train/common.py)
